# 01 · Prompt와 모델

**목표**: Gen AI Hub SDK로 LLM을 호출하고, message role과 system instruction이 출력에 미치는 영향을 비교합니다.

LLM은 다음 토큰을 예측하는 모델이며, 우리가 보내는 입력 전체를 prompt라고 합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Prompt Chatbot (UI)</h2>
<p style="margin: 7px 0;">아래 UI에서 system prompt와 모델을 바꾸며 대화를 이어가세요. 대화 history는 현재 브라우저 session에만 유지되며, AI Core에는 답변 생성 요청만 보냅니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;">
<li>질문을 입력하고 답변을 확인합니다.</li>
<li>⚙️ 설정에서 system prompt 또는 모델을 바꿔 응답 형식을 비교합니다.</li>
<li>이 UI는 업무 상태를 변경하지 않습니다.</li>
</ol>
  </div>
</div>


In [13]:
from pathlib import Path
import json
from time import perf_counter
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path[:0] = [str(project_root / 'shared'), str(project_root / '2_gen_ai_hub_llm')]
from src.lab_core import run_chat, run_chat_stream
from lab_common.gradio_ui import build_chat_ui
chat_ui = build_chat_ui(run_chat, chat_stream_runner=run_chat_stream)
chat_ui.launch(inline=True, share=False, server_name='127.0.0.1', show_error=True, width='100%', height=820)

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">메시지 role과 준비</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li><code>system</code>은 모델의 역할, 규칙, 언어, 출력 형식을 정합니다.</li>
<li><code>user</code>는 사용자의 질문 또는 요청입니다.</li>
<li><code>assistant</code>는 이전 모델 응답으로, 대화 history에 사용합니다.</li>
</ul>
<p style="margin: 7px 0;">아래 코드는 루트 <code>.env</code>의 <code>LLM_MODEL</code>을 사용합니다. UI를 실행한 뒤에는 선택 심화 자료로 코드 기반 비교를 확인하세요.</p>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


In [14]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path[:0] = [str(project_root / 'shared'), str(project_root / '2_gen_ai_hub_llm')]
from lab_common.ai_core import chat_text
from lab_common.config import load_settings
from src.prompt_model.lab_core import compare_prompts, run_chat
settings = load_settings()

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 prompt 비교하기</h2>
<p style="margin: 7px 0;">이 아래 셀들은 UI와 같은 LLM 호출을 공개 코어 <code>compare_prompts()</code>로 확인하는 심화 자료입니다. 기본 실습에는 실행할 필요가 없습니다.</p>
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">기본 chat 응답</h3>
<p style="margin: 7px 0;">같은 질문을 기본 조건과 system 조건으로 함께 호출하고, 먼저 기본 응답을 확인합니다.</p>
  </div>
</div>


In [15]:
question = 'LLM의 기본 아키텍처를 한 문장으로 설명해줘.'
comparison = compare_prompts(question, settings)
print(comparison['baseline'])

LLM의 기본 아키텍처는 대규모 데이터로 학습된 트랜스포머(Transformer) 기반 신경망 모델입니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">system prompt로 제약하기</h3>
<p style="margin: 7px 0;">같은 코어 호출에서 반환된 system prompt 적용 결과를 확인합니다. system prompt는 말투뿐 아니라 대상 독자, 언어, 문장 수, 표나 JSON 같은 출력 형식도 제어할 수 있습니다.</p>
  </div>
</div>


In [16]:
print(comparison['constrained'])

LLM의 기본 아키텍처는 대규모 텍스트 데이터를 입력받아 확률적으로 다음 단어를 예측하는 다층 트랜스포머 신경망입니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 2</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">message 배열을 직접 구성하기</h2>
<p style="margin: 7px 0;">LLM API에는 하나의 문자열이 아니라 역할이 붙은 메시지 배열을 보냅니다. 아래에서 <code>system</code> 규칙, 이전 <code>user</code> 질문, 이전 <code>assistant</code> 답변, 새 <code>user</code> 질문의 순서를 확인하고 실행하세요.</p>
  </div>
</div>


In [17]:
messages = [
    {"role": "system", "content": "너는 친절한 교육 보조자다. 한국어로 간결하게 답한다."},
    {"role": "user", "content": "LLM의 장점을 두 가지 알려줘."},
    {"role": "assistant", "content": "자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다."},
    {"role": "user", "content": "방금 답을 표로 정리해줘."},
]
for index, item in enumerate(messages, start=1):
    print(f'{index}. {item["role"]}: {item["content"]}')

reply = chat_text(settings, messages, settings.llm_model)
print()
print('응답:')
print(reply)

1. system: 너는 친절한 교육 보조자다. 한국어로 간결하게 답한다.
2. user: LLM의 장점을 두 가지 알려줘.
3. assistant: 자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다.
4. user: 방금 답을 표로 정리해줘.

응답:
| 장점 번호 | 장점 내용                                 |
|------------|-------------------------------------------|
| 1          | 자연어를 이해할 수 있음                   |
| 2          | 다양한 텍스트 작업을 도울 수 있음         |


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 3</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">대화 history가 맥락을 전달하는 방식</h2>
<p style="margin: 7px 0;">두 번째 질문만 보면 무엇을 표로 바꾸라는지 알 수 없습니다. 같은 질문을 history 없이, 그리고 history와 함께 보내 결과를 비교해 보세요.</p>
<p style="margin: 7px 0;"><strong>관찰:</strong> 코어는 최근 12개 메시지와 총 12,000자 안에서만 history를 보존합니다. 긴 대화에서는 중요한 맥락을 요약하거나 문서 검색으로 보완해야 합니다.</p>
  </div>
</div>


In [18]:
follow_up = '방금 답을 표로 정리해줘.'
history = [
    {"role": "user", "content": "LLM의 장점을 두 가지 알려줘."},
    {"role": "assistant", "content": "자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다."},
]
without_history = run_chat(follow_up, settings)
with_history = run_chat(follow_up, settings, history=history)
print('[history 없음]')
print(without_history['reply'])
print()
print('[history 포함]')
print(with_history['reply'])

[history 없음]
물론입니다! 하지만 바로 전에 질문이나 답변 내용이 보이지 않습니다. 이전 답변 내용을 표로 정리해드리려면 어떤 내용을 표로 만들지 말씀해주실 수 있을까요?  
혹시 이전 답변을 다시 입력해주시면 표로 정리해드리겠습니다.

[history 포함]
| 장점 번호 | LLM의 장점                                    |
|------------|----------------------------------------------|
| 1          | 자연어를 이해할 수 있다.                     |
| 2          | 다양한 텍스트 작업(요약, 번역 등)을 도울 수 있다. |


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 4</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">프롬프트로 JSON 형식 요청하고 검증하기</h2>
<p style="margin: 7px 0;">형식을 요청해도 항상 파싱 가능한 JSON이 반환된다고 보장할 수는 없습니다. 실행 결과가 파싱에 성공하는지 확인하고, 실패했다면 모델이 반환한 원문을 살펴보세요.</p>
  </div>
</div>


In [19]:
json_prompt = (
    '너는 교육 보조자다. 반드시 JSON 객체만 반환한다. Markdown 코드 블록이나 설명을 덧붙이지 않는다. '
    '키는 summary(문자열), keywords(문자열 배열)만 사용한다.'
)
json_outcome = run_chat(
    'LLM을 한 문장으로 요약하고 핵심 키워드 세 개를 추출해줘.',
    settings,
    system_prompt=json_prompt,
)
try:
    parsed = json.loads(json_outcome['reply'])
    print('JSON 파싱 성공:', parsed)
except json.JSONDecodeError:
    print('JSON 파싱 실패. 반환 원문을 확인하세요:')
    print(json_outcome['reply'])

JSON 파싱 성공: {'summary': 'LLM은 인간의 언어를 이해하고 생성하는 대규모 인공지능 언어 모델이다.', 'keywords': ['인공지능', '언어모델', '자연어처리']}


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 실험</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">동일 질문을 모델별로 비교하기</h2>
<p style="margin: 7px 0;">현재 모델과 AI Core에 배포된 다른 모델을 같은 질문으로 호출해 보세요. 응답 시간뿐 아니라 정확성, 지시 준수, 출력 형식을 함께 기록합니다. 배포되지 않은 모델명은 호출할 수 없습니다.</p>
  </div>
</div>


In [20]:
models_to_compare = [settings.llm_model]
# AI Core에 RUNNING 상태로 배포된 모델이 있다면 아래처럼 추가합니다.
# models_to_compare.append('배포된-다른-모델명')

comparison_question = 'LLM의 한계를 초보 개발자에게 두 문장으로 설명해줘.'
for model_name in models_to_compare:
    started_at = perf_counter()
    outcome = run_chat(comparison_question, settings, model=model_name)
    elapsed = perf_counter() - started_at
    print(f'[{model_name}] {elapsed:.1f}초')
    print(outcome['reply'])
    print()

[gpt-4.1] 2.8초
LLM(대형 언어 모델)은 기존 데이터에 기반해 답변을 생성하므로 틀린 정보를 제공할 수 있습니다. 또한, 맥락이나 실제 상황을 완전히 이해하지 못해 항상 인간처럼 사고하거나 판단하지는 못합니다.



<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 미션</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">사내 FAQ 답변 도우미의 system prompt 설계</h2>
<p style="margin: 7px 0;">아래 셀에서 대상 독자, 말투, 답변 형식, 근거가 부족할 때의 행동을 포함한 system prompt를 작성하세요. 같은 질문을 기본 응답과 비교하고, 요구한 형식과 안전 규칙이 지켜졌는지 평가합니다.</p>
  </div>
</div>


In [21]:
faq_system_prompt = '''
TODO: 사내 FAQ 답변 도우미의 규칙을 작성하세요.
'''
faq_question = '출장비 정산 마감일이 언제인가요?'

# 작성 후 아래 주석을 해제해 기본 응답과 비교하세요.
# print(run_chat(faq_question, settings)['reply'])
# print(run_chat(faq_question, settings, system_prompt=faq_system_prompt)['reply'])